# 03. Robot Motion Models

Motion model은 명령 $u_t$가 들어왔을 때 로봇 상태가 어떻게 변할지 확률적으로 표현한다.

$$p(x_t\mid u_t,x_{t-1})$$

실제 바퀴는 미끄러지고 encoder는 노이즈가 있으므로, motion model은 deterministic kinematics + noise로 본다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os
os.makedirs('assets', exist_ok=True)

for font_name in ['Nanum Gothic', 'AppleGothic', 'Malgun Gothic']:
    if any(font.name == font_name for font in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['axes.unicode_minus'] = False

## 1. Velocity Motion Model 샘플링

입력 $u=[v,\omega]$에 노이즈를 넣고 unicycle model로 pose를 샘플링한다.

In [ ]:
np.random.seed(3)
def step_unicycle(x, v, w, dt):
    if abs(w) < 1e-6:
        return x + np.array([v*dt*np.cos(x[2]), v*dt*np.sin(x[2]), 0])
    return x + np.array([
        -v/w*np.sin(x[2]) + v/w*np.sin(x[2]+w*dt),
         v/w*np.cos(x[2]) - v/w*np.cos(x[2]+w*dt),
         w*dt
    ])

def sample_motion_velocity(x, u, dt, alphas, n=500):
    v, w = u
    samples=[]
    for _ in range(n):
        vh = v + np.random.randn()*np.sqrt(alphas[0]*v*v + alphas[1]*w*w)
        wh = w + np.random.randn()*np.sqrt(alphas[2]*v*v + alphas[3]*w*w)
        xn = step_unicycle(x, vh, wh, dt)
        xn[2] = np.arctan2(np.sin(xn[2]), np.cos(xn[2]))
        samples.append(xn)
    return np.array(samples)

x0=np.array([0.0,0.0,np.deg2rad(20)])
samples=sample_motion_velocity(x0, [1.0,0.7], 1.2, [0.05,0.02,0.02,0.08], n=1000)
nominal=step_unicycle(x0,1.0,0.7,1.2)
fig, ax=plt.subplots(figsize=(7,6))
ax.scatter(samples[:,0],samples[:,1],s=8,color='#534AB7',alpha=0.25,label='sampled next poses')
ax.scatter(nominal[0],nominal[1],color='#E85D24',s=100,label='noise-free')
ax.scatter(x0[0],x0[1],color='black',s=80,label='start')
ax.set_aspect('equal'); ax.grid(alpha=0.25); ax.legend(); ax.set_title('Velocity motion model samples')
plt.savefig('assets/03_velocity_motion_samples.png',dpi=150,bbox_inches='tight'); plt.show()
print('sample mean:', np.round(samples.mean(axis=0),3))
print('sample covariance xy:')
print(np.round(np.cov(samples[:,:2].T),4))

## 2. Odometry Motion Model 직관

Odometry update는 보통 회전-이동-회전으로 분해한다.

$$u_t=(\delta_{rot1},\delta_{trans},\delta_{rot2})$$

In [ ]:
np.random.seed(4)
def sample_odometry(x, odom, alphas, n=800):
    r1, trans, r2 = odom
    out=[]
    for _ in range(n):
        hr1 = r1 + np.random.randn()*np.sqrt(alphas[0]*r1*r1 + alphas[1]*trans*trans)
        ht = trans + np.random.randn()*np.sqrt(alphas[2]*trans*trans + alphas[3]*(r1*r1+r2*r2))
        hr2 = r2 + np.random.randn()*np.sqrt(alphas[0]*r2*r2 + alphas[1]*trans*trans)
        out.append([x[0]+ht*np.cos(x[2]+hr1), x[1]+ht*np.sin(x[2]+hr1), x[2]+hr1+hr2])
    return np.array(out)

odom=(np.deg2rad(25),1.4,np.deg2rad(-10))
osamps=sample_odometry(np.array([0,0,0.0]), odom, [0.03,0.02,0.04,0.01])
fig, ax=plt.subplots(figsize=(7,6))
ax.scatter(osamps[:,0],osamps[:,1],s=8,color='#1D9E75',alpha=0.25)
for i in range(0,len(osamps),80):
    ax.arrow(osamps[i,0],osamps[i,1],0.12*np.cos(osamps[i,2]),0.12*np.sin(osamps[i,2]),color='gray',alpha=0.4,head_width=0.025)
ax.set_aspect('equal'); ax.grid(alpha=0.25); ax.set_title('Odometry motion model: rot-trans-rot noise')
plt.savefig('assets/03_odometry_motion_samples.png',dpi=150,bbox_inches='tight'); plt.show()

## 요약

| 모델 | 입력 | 책 커리큘럼 연결 |
|------|------|------------------|
| Velocity model | $v,\omega$ | Ch.5 Robot Motion |
| Odometry model | rot-trans-rot | Ch.5 Robot Motion |
| Sampling | 확률분포에서 다음 pose 생성 | Particle filter / MCL의 prediction 단계 |